# Multi-LLM Sequential Workflow: OpenAI → Claude → Gemini

This hands-on lab demonstrates how three different LLMs can work together in sequence:

1. **OpenAI** creates a challenging technical architecture question.
2. **Claude** analyzes the question and produces a solution.
3. **Gemini** explains Claude's solution in beginner-friendly detail.

> **Important:** This is a sequential LLM workflow (orchestration), not a fully autonomous agent system. You need API keys for all three providers. API usage may incur charges.

## 1. Install the required packages

Run this cell once in your notebook environment.

In [ ]:
%pip install -q openai anthropic google-genai python-dotenv

## 2. Configure your API keys

Create a file named `.env` in the same folder as this notebook and add your keys:

```text
OPENAI_API_KEY=your_openai_api_key
ANTHROPIC_API_KEY=your_anthropic_api_key
GEMINI_API_KEY=your_gemini_api_key
```

Keep this file private. **Do not paste API keys directly into a notebook that you plan to share or upload to GitHub.**

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

missing = [
    name for name, value in {
        "OPENAI_API_KEY": OPENAI_API_KEY,
        "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
        "GEMINI_API_KEY": GEMINI_API_KEY,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        "Missing API key(s): " + ", ".join(missing) +
        ". Add them to your .env file and rerun this cell."
    )

print("All three API keys were loaded successfully.")

## 3. Import SDKs and initialize clients

The model names below are examples. If a model is unavailable to your account, replace it with a model ID you can access. You can also set model names in your `.env` file.

In [ ]:
from openai import OpenAI
from anthropic import Anthropic
from google import genai

openai_client = OpenAI(api_key=OPENAI_API_KEY)
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-opus-5-5")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")

print("Clients initialized.")
print("OpenAI model:", OPENAI_MODEL)
print("Claude model:", CLAUDE_MODEL)
print("Gemini model:", GEMINI_MODEL)

## 4. Stage 1 — OpenAI generates a complex question

OpenAI is asked to create a realistic, senior-level system design problem. The output becomes the input for Claude.

In [1]:
question_prompt = """
Create one challenging, realistic system-design question for a Senior Java
Solution Architect interview.

The problem should involve a high-throughput, cloud-native microservices
platform using Java and Spring Boot. Include requirements around scalability,
availability, security, data consistency, messaging, observability, and
deployment. Provide enough business context and measurable constraints for
a candidate to design a complete solution.

Return only the interview question, with clearly stated requirements.
"""

question_response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input=question_prompt
)

architecture_question = question_response.output_text
print(architecture_question)

NameError: name 'openai_client' is not defined

## 5. Stage 2 — Claude solves the question

Claude receives the exact question generated by OpenAI and prepares a structured architecture solution.

In [ ]:
claude_prompt = f"""
You are a principal software architect with deep expertise in Java,
Spring Boot, microservices, Kafka, distributed systems, and cloud architecture.

Solve the following system-design problem. Structure your response with:
1. Requirements and assumptions
2. High-level architecture and component responsibilities
3. Key request and event flows
4. Data storage and consistency strategy
5. Scalability, resilience, and failure handling
6. Security and observability
7. Deployment and CI/CD approach
8. Trade-offs and alternatives

Be technically detailed and explain why you recommend each major choice.

QUESTION:
{architecture_question}
"""

claude_response = claude_client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=6000,
    messages=[{"role": "user", "content": claude_prompt}]
)

claude_solution = "\n".join(
    block.text for block in claude_response.content
    if getattr(block, "type", None) == "text"
)

print(claude_solution)

## 6. Stage 3 — Gemini explains Claude's solution

Gemini receives both the original question and Claude's answer. It explains the design step by step, assuming the reader is learning the concepts.

In [ ]:
gemini_prompt = f"""
Act as an expert technical instructor teaching a Java developer who is
learning system design and cloud-native architecture.

Explain Claude's proposed solution in detail. Assume the learner knows
Java and Spring Boot but may be new to some distributed-systems concepts.

For each major component:
- Explain what it does in simple language.
- Explain why it is needed.
- Walk through how data moves through the system.
- Explain important design decisions and trade-offs.
- Give practical examples where useful.

Finish with a glossary of important technical terms and five questions
the learner can use to test their understanding.

ORIGINAL QUESTION:
{architecture_question}

CLAUDE'S SOLUTION:
{claude_solution}
"""

gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=gemini_prompt
)

gemini_explanation = gemini_response.text
print(gemini_explanation)

## 7. Save the complete workflow as a Markdown report

This combines the question, Claude's solution, and Gemini's explanation into one report.

In [ ]:
from datetime import datetime
from pathlib import Path

report = f"""# Multi-LLM Architecture Workflow

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Stage 1 — Question generated by OpenAI

{architecture_question}

## Stage 2 — Solution from Claude

{claude_solution}

## Stage 3 — Explanation from Gemini

{gemini_explanation}
"""

filename = f"multi_llm_report_{datetime.now():%Y%m%d_%H%M%S}.md"
Path(filename).write_text(report, encoding="utf-8")

print(f"Report saved to: {filename}")

## 8. Student exercises

1. Change OpenAI's prompt so it generates a question about an e-commerce platform.
2. Ask Claude to provide a Mermaid architecture diagram in its answer.
3. Ask Gemini to explain Kafka partitions, consumer groups, and retry handling from the proposed design.
4. Add a fourth step in which OpenAI reviews Claude's solution and identifies gaps.
5. Compare the outputs from different model choices.

### Troubleshooting

- **Authentication error:** Check that all three API keys are correct and loaded from `.env`.
- **Model not found / access denied:** Replace the model ID with one currently available to your account.
- **Rate limit or quota error:** Check your provider's usage limits and billing.
- **Package import error:** Rerun the installation cell, then restart the notebook kernel if needed.

### Workflow summary

`OpenAI (generate question) → Claude (solve) → Gemini (explain) → Save report`